# 📊 Step 1: Advanced Data Parsing (Markdown & Text Chunking)

GraphRAG의 첫 번째 단계는 더러운 원본 텍스트를 LLM이 소화하기 좋은 '의미 있는 덩어리(Chunk)'로 쪼개는 것입니다.
이 노트북에서는 수집된 미국 주식/투자 마크다운 문서를 헤더(Header) 기준으로 파싱하여 구조화된 딕셔너리로 변환합니다.

In [1]:
import re
import json

def parse_markdown_sources(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
        
    # '## 숫자️⃣ 소스명 - 제목' 형태의 정규식 매칭
    pattern = r'## \d+️⃣ (.*?) - (.*?)\n(.*?)(?=\n## |\Z)'
    matches = re.finditer(pattern, text, re.DOTALL)
    
    parsed_data = []
    for match in matches:
        source_name = match.group(1).strip()
        title = match.group(2).strip()
        content = match.group(3).strip().replace('\n', ' ')
        
        # 불필요한 마크다운 기호 제거 (--- 등)
        content = re.sub(r'-{3,}', '', content).strip()
        
        parsed_data.append({
            'source': source_name,
            'title': title,
            'content': content
        })
        
    return parsed_data

# 파싱 실행
file_path = '../data/sample_stock_data.md'
chunks = parse_markdown_sources(file_path)

print(f"총 {len(chunks)}개의 소스가 성공적으로 파싱되었습니다!\n")
for i, chunk in enumerate(chunks[:2]): # 처음 2개만 확인
    print(f"[{i+1}] 소스: {chunk['source']}")
    print(f"    제목: {chunk['title']}")
    print(f"    내용: {chunk['content'][:100]}...\n")

총 8개의 소스가 성공적으로 파싱되었습니다!

[1] 소스: 수페TV
    제목: 나스닥 반등 vs 금값
    내용: 나스닥이 반등 신호를 보이고 있으나 기저 약세 요인이 여전히 존재. 금값은 지정학적 리스크(호르무즈 해협, 중동 정세)와 인플레이션 헤지 수요로 강세 지속. 반도체 섹터는 기술주 ...

[2] 소스: mijooeun
    제목: 주식 매매 기술적분석
    내용: RSI(상대강도지수)와 모멘텀 지표를 활용한 매매 전략의 실행 가능성 검토. 50일 이평선 이탈이 추세 변화의 선행 신호로 작용하며, 거래량 급증 후 가격 확인 국면이 중요한 진입...



## 💾 파싱된 데이터 저장
추출된 청크를 JSON 형태로 저장하여 다음 단계(Phase 2: 지식 그래프 추출)에서 활용할 수 있게 준비합니다.

In [2]:
output_path = '../data/parsed_chunks.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=4)

print(f"성공적으로 파싱 데이터가 저장되었습니다: {output_path}")

성공적으로 파싱 데이터가 저장되었습니다: ../data/parsed_chunks.json
